In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request
import matplotlib.pyplot as plt
#%run WIRE_ISSUES_MODULE.py
import warnings
warnings.filterwarnings("ignore")





db = client.get_database('telematics_db')


db2 = client2.get_database('telematics_db')







In [ ]:
def loc_pings(l):
    try:
        
        vid=l[0]
        day_start=l[1]
        day_end=l[2]

        try:
            url=GPS_API_BASE+"/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(vid,int(day_start),int(day_end))
           # print(url)
            data=json.load(urllib.request.urlopen(url))
            df=pd.DataFrame(data['data'][str(vid)])
            #print("LOC PINGS A GAYI")

        except Exception as e:
            print(" loc pings issue >> " + str(e))
            df=pd.DataFrame()
        return df
    except:
        print("check the values being passed for loc_pings")

In [ ]:
def hb_pings(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]
    
    try:        
        url=GPS_API_BASE+"/heartbeat/list?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(vid,int(day_start),int(day_end))
        hb_data=json.load(urllib.request.urlopen(url))
        hb=pd.DataFrame(hb_data['data'][str(vid)])
        #print("HB AAYA")

    except Exception as e:

#        print("hb pings issue >> " + str(e))
        hb=pd.DataFrame()
        
    return hb

In [ ]:
def device_history(vid,start,end):
    try:
        
        hb=hb_pings([vid,start,end])
        lc=loc_pings([vid,start,end])
        device_histories=[]
        if len(hb)>0:
            hb.sort_values('created',inplace=True)
            hb['created']=hb['created']/1000
            hb['created_shift']=hb['created'].shift(1)
            hb['created_diff']=hb['created']-hb['created_shift']
            no_infos=hb[hb['created_diff']>3600]
            if len(no_infos)>0:
                for i,row in no_infos.iterrows():
                    device_histories.append([row['vehicleId'],row['created'],row['created_shift']])
        elif (len(hb)==0)&(len(lc)==0):
            print('noData')

        else:
            lc.sort_values('created',inplace=True)
            lc['created']=lc['created']/1000
            lc['created_shift']=lc['created'].shift(1)
            lc['created_diff']=lc['created']-lc['created_shift']
            no_infos=lc[lc['created_diff']>3600]
            if len(no_infos)>0:
                for i,row in no_infos.iterrows():
                    device_histories.append([row['vehicleId'],row['created'],row['created_shift']])
        if len(device_histories)>0:
            return device_histories
        else:
            return None
    except Exception as e:
        print("issue with device_history is :"+str(e))

    
    
    
